In [ ]:
# Línea 1: módulos del sistema, análisis de datos y colecciones
import os                           # Manejo de rutas y carpetas del sistema operativo
import sys                          # Permite detectar si se está ejecutando en Colab
from collections import Counter     # Cuenta frecuencias de elementos en una lista
import pandas as pd                 # Tablas y DataFrames para mostrar resultados

# Línea 2: configuración del backend gráfico según el entorno de ejecución
import matplotlib
if 'google.colab' not in sys.modules:
    matplotlib.use('Agg')           # Modo sin ventana cuando no se ejecuta en Colab

# Línea 3: importar el módulo principal de gráficas
import matplotlib.pyplot as plt

# Línea 4: ajustes visuales globales para todas las figuras del proyecto
plt.rcParams.update({
    'figure.dpi'        : 120,      # Resolución en puntos por pulgada
    'axes.spines.top'   : False,    # Quita el borde superior de cada gráfico
    'axes.spines.right' : False,    # Quita el borde derecho de cada gráfico
    'axes.titlesize'    : 11,       # Tamaño de fuente para los títulos
    'axes.labelsize'    : 9,        # Tamaño de fuente para las etiquetas de los ejes
})

# Línea 5: carpeta de salida para las imágenes y constante global infinito
RESULTADOS = os.path.join(os.getcwd(), 'resultados')  # Ruta donde se guardarán las imágenes
os.makedirs(RESULTADOS, exist_ok=True)                 # Crea la carpeta si todavía no existe
INF = float('inf')                                     # Representa ausencia de techo máximo

In [ ]:
# Pool de 20 estudiantes disponibles para la selección
# dict(zip()) convierte cada tupla en un diccionario con claves nombradas
POOL = [
    dict(zip('id programa semestre ciudad jornada'.split(), r))
    for r in [
        #  id    programa        sem    ciudad         jornada
        (  1,   'Computacion',   1,    'Manizales',   'Diurna'  ),
        (  2,   'Matematicas',   1,    'Pereira',     'Diurna'  ),
        (  3,   'Estadistica',   2,    'Armenia',     'Nocturna'),
        (  4,   'Computacion',   2,    'Manizales',   'Diurna'  ),
        (  5,   'Matematicas',   3,    'Pereira',     'Nocturna'),
        (  6,   'Estadistica',   1,    'Armenia',     'Diurna'  ),
        (  7,   'Fisica',        3,    'Manizales',   'Diurna'  ),
        (  8,   'Computacion',   1,    'Pereira',     'Nocturna'),
        (  9,   'Matematicas',   4,    'Manizales',   'Diurna'  ),
        ( 10,   'Estadistica',   2,    'Pereira',     'Diurna'  ),
        ( 11,   'Computacion',   3,    'Armenia',     'Nocturna'),
        ( 12,   'Fisica',        1,    'Manizales',   'Diurna'  ),
        ( 13,   'Matematicas',   2,    'Armenia',     'Nocturna'),
        ( 14,   'Computacion',   4,    'Pereira',     'Diurna'  ),
        ( 15,   'Estadistica',   3,    'Manizales',   'Nocturna'),
        ( 16,   'Fisica',        2,    'Armenia',     'Diurna'  ),
        ( 17,   'Computacion',   1,    'Armenia',     'Diurna'  ),
        ( 18,   'Matematicas',   5,    'Pereira',     'Nocturna'),
        ( 19,   'Estadistica',   4,    'Manizales',   'Diurna'  ),
        ( 20,   'Fisica',        5,    'Armenia',     'Nocturna'),
    ]
]

print(f'Pool cargado: {len(POOL)} estudiantes')
print(pd.DataFrame(POOL).to_string(index=False))

In [ ]:
# Conjuntos de cuotas: definen las restricciones que debe cumplir la muestra

C1 = {
    'nombre'         : 'Conjunto 1 — Base',
    'tamano'         : 6,                                     # La muestra debe tener exactamente 6 estudiantes
    'min_programa'   : {'Computacion': 2, 'Matematicas': 1},  # Mínimos obligatorios por carrera
    'max_por_ciudad' : 3,                                     # Máximo 3 estudiantes de la misma ciudad
    'min_semestre_1' : 2,                                     # Al menos 2 estudiantes de primer semestre
    'min_nocturna'   : 0,                                     # Sin restricción de jornada nocturna
}

C2 = {
    'nombre'         : 'Conjunto 2 — Alta diversidad',
    'tamano'         : 6,
    'min_programa'   : {'Computacion': 2, 'Matematicas': 1, 'Estadistica': 1},
    'max_por_ciudad' : 3,
    'min_semestre_1' : 1,
    'min_nocturna'   : 2,                                     # Requiere al menos 2 estudiantes nocturnos
}

C3 = {
    'nombre'         : 'Conjunto 3 — Sin solución',
    'tamano'         : 7,                                     # Se piden 7, pero solo caben 6: imposible
    'min_programa'   : {'Computacion': 2, 'Matematicas': 1},
    'max_por_ciudad' : 2,                                     # 2 × 3 ciudades = 6 < 7 requeridos
    'min_semestre_1' : 2,
    'min_nocturna'   : 0,
}

In [ ]:
# _cnt: cuenta los atributos actuales de la muestra parcial
def _cnt(m):
    por_programa = Counter(e['programa'] for e in m)        # Cuántos hay de cada carrera
    por_ciudad   = Counter(e['ciudad']   for e in m)        # Cuántos hay de cada ciudad
    por_jornada  = Counter(e['jornada']  for e in m)        # Cuántos hay en cada jornada
    sem1         = sum(1 for e in m if e['semestre'] == 1)  # Total de estudiantes de semestre 1
    return por_programa, por_ciudad, por_jornada, sem1


# _podar: decide si una rama del árbol es inviable antes de explorarla
def _podar(m, r, q):
    faltantes = q['tamano'] - len(m)          # Cuántos estudiantes aún faltan en la muestra
    p, c, j, s = _cnt(m)                     # Conteos actuales de la muestra parcial
    mc = q.get('max_por_ciudad', INF)         # Techo máximo de estudiantes por ciudad

    if mc < INF:                              # Condición 0: capacidad total de ciudades insuficiente
        todas = set(e['ciudad'] for e in r) | set(c)
        capacidad = sum(max(0, mc - c.get(x, 0)) for x in todas)  # Cupos disponibles en total
        if capacidad < faltantes:
            return True

    ef = [e for e in r if c.get(e['ciudad'], 0) < mc] if mc < INF else r  # Candidatos que no exceden el techo

    if len(ef) < faltantes:                   # Condición 1: no hay suficientes candidatos efectivos
        return True

    if any(p.get(k, 0) + sum(1 for e in ef if e['programa'] == k) < v    # Condición 2: mínimo de programa inalcanzable
           for k, v in q['min_programa'].items()):
        return True

    if s + sum(1 for e in ef if e['semestre'] == 1) < q['min_semestre_1']:  # Condición 3: semestre 1 insuficiente
        return True

    if j.get('Nocturna', 0) + sum(1 for e in ef if e['jornada'] == 'Nocturna') < q['min_nocturna']:  # Condición 4
        return True

    return False  # Ninguna condición activada: la rama sigue siendo viable

In [ ]:
# backtracking: construye la muestra eligiendo, explorando y retrocediendo
def backtracking(pool, m, q, sols, cnt, i0=0, lim=1):
    cnt['llamadas_recursivas'] += 1          # Registra cada entrada a la función

    if len(m) == q['tamano']:                # Caso base: la muestra alcanzó el tamaño requerido
        p, _, j, s = _cnt(m)
        cumple = (
            all(p.get(k, 0) >= v for k, v in q['min_programa'].items())  # Verifica mínimos de programa
            and s >= q['min_semestre_1']                                   # Verifica mínimo semestre 1
            and j.get('Nocturna', 0) >= q['min_nocturna']                 # Verifica mínimo nocturna
        )
        if cumple:
            sols.append(list(m))             # Guarda una copia de la solución válida encontrada
        return

    if lim and len(sols) >= lim:             # Ya se alcanzó el número de soluciones pedido
        return

    r = pool[i0:]                            # Candidatos disponibles desde la posición i0

    if _podar(m, r, q):                      # La rama es inviable: no explorar más
        cnt['ramas_descartadas'] += 1
        return

    mc  = q.get('max_por_ciudad', INF)
    ciu = Counter(e['ciudad'] for e in m)    # Ciudades ya representadas en la muestra actual

    for i, e in enumerate(r):
        if lim and len(sols) >= lim:
            return
        if ciu.get(e['ciudad'], 0) < mc:     # El candidato no excede el techo de su ciudad
            m.append(e)                      # ELEGIR: agregar el candidato a la muestra
            backtracking(pool, m, q, sols, cnt, i0 + i + 1, lim)  # EXPLORAR: llamada recursiva
            cnt['retrocesos'] += 1
            m.pop()                          # RETROCEDER: quitar el candidato y probar el siguiente
        else:
            cnt['ramas_descartadas'] += 1    # El candidato viola max_por_ciudad: descartar

In [ ]:
# buscar: función de entrada al algoritmo, imprime resultados y estadísticas
def buscar(q, pool=None, lim=1, v=True):
    pool = pool or POOL                      # Usa el pool global si no se indica otro
    sols = []                                # Lista donde se acumulan las soluciones encontradas
    cnt  = {
        'llamadas_recursivas': 0,            # Total de veces que se entró a backtracking
        'ramas_descartadas'  : 0,            # Ramas eliminadas por restricciones o poda
        'retrocesos'         : 0,            # Veces que se deshizo una decisión
    }

    backtracking(pool, [], q, sols, cnt, lim=lim)   # Inicia el algoritmo con muestra vacía

    if not v:
        return sols, cnt                     # Retorno silencioso para uso interno

    print()
    print('=' * 55)
    print(f"  {q['nombre']}")
    print(f"  Tamaño              : {q['tamano']}")
    print(f"  Llamadas recursivas : {cnt['llamadas_recursivas']}")
    print(f"  Ramas descartadas   : {cnt['ramas_descartadas']}")
    print(f"  Retrocesos          : {cnt['retrocesos']}")
    print(f"  Soluciones halladas : {len(sols)}")

    if sols:
        print(pd.DataFrame(sols[0]).to_string(index=False))   # Muestra la primera solución como tabla
        p, _, j, z = _cnt(sols[0])
        print(f"  Programas  : {dict(p)}")
        print(f"  Semestre 1 : {z} >= {q['min_semestre_1']}")
        print(f"  Nocturna   : {j.get('Nocturna', 0)} >= {q['min_nocturna']}")
    else:
        cap = q['max_por_ciudad'] * 3        # Capacidad máxima real: techo × número de ciudades
        print(f"  *** SIN SOLUCION *** {q['max_por_ciudad']}x3 = {cap} < {q['tamano']}")

    return sols, cnt

In [ ]:
# Ejecución del algoritmo para los tres conjuntos de cuotas
s1, c1 = buscar(C1)   # Caso base: tiene solución
s2, c2 = buscar(C2)   # Alta diversidad: más restricciones, tiene solución
s3, c3 = buscar(C3)   # Sin solución: la poda lo detecta en la primera llamada

In [ ]:
# Gráfica comparativa: distribución del pool original vs. muestra seleccionada
fig, axes = plt.subplots(2, 3, figsize=(14, 8))                                    # 2 filas (C1, C2) × 3 columnas
fig.suptitle('Pool original vs. Muestras seleccionadas', fontsize=13, fontweight='bold')

for row, (m, q, col) in enumerate([(s1[0], C1, '#4472C4'), (s2[0], C2, '#ED7D31')]):
    for ax, f in zip(axes[row], ['programa', 'ciudad', 'jornada']):                # Una columna por dimensión
        cp   = Counter(e[f] for e in POOL)     # Distribución del pool completo
        cm   = Counter(e[f] for e in m)        # Distribución de la muestra seleccionada
        cats = sorted(cp)                      # Categorías ordenadas alfabéticamente
        x    = range(len(cats))
        w    = 0.35                            # Ancho de cada barra individual

        ax.bar([i - w/2 for i in x], [cp[c] for c in cats],
               w, color='#4472C4', alpha=.85, label=f'Pool ({len(POOL)})')         # Barras del pool
        ax.bar([i + w/2 for i in x], [cm.get(c, 0) for c in cats],
               w, color='#ED7D31', alpha=.85, label=f'Muestra ({len(m)})')         # Barras de la muestra

        ax.set_xticks(list(x))
        ax.set_xticklabels(cats, rotation=25, ha='right', fontsize=8)
        ax.set_title(f'Por {f}', fontsize=9)
        ax.set_ylabel('N.°', fontsize=8)
        ax.legend(fontsize=7)

    axes[row][0].annotate(                                                          # Etiqueta del conjunto en la fila
        q['nombre'], xy=(.5, 1.12), xycoords='axes fraction',
        ha='center', fontsize=9, fontweight='bold', color=col
    )

plt.tight_layout()
plt.savefig(f'{RESULTADOS}/comparacion_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfica de eficiencia: llamadas recursivas y ramas descartadas por conjunto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Eficiencia del backtracking', fontsize=12, fontweight='bold')

metricas = [
    (ax1, [c1['llamadas_recursivas'], c2['llamadas_recursivas'], c3['llamadas_recursivas']], 'Llamadas recursivas'),
    (ax2, [c1['ramas_descartadas'],   c2['ramas_descartadas'],   c3['ramas_descartadas']  ], 'Ramas descartadas'),
]

for ax, vals, titulo in metricas:
    ax.bar(['C1', 'C2', 'C3'], vals,
           color=['#4472C4', '#ED7D31', '#A9D18E'], alpha=.87, edgecolor='white')  # Un color por conjunto
    ax.set(title=titulo, ylabel='Cantidad')
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals) * .01, str(v), ha='center', fontsize=9, fontweight='bold')  # Valor sobre cada barra

plt.tight_layout()
plt.savefig(f'{RESULTADOS}/eficiencia_backtracking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# balance: mide qué tan bien la muestra representa al pool en distribución de programas
def balance(m, pool=POOL):
    n  = len(pool)                                   # Total de estudiantes en el pool
    k  = len(m)                                      # Total de estudiantes en la muestra
    cp = Counter(e['programa'] for e in pool)        # Proporción de programas en el pool
    cm = Counter(e['programa'] for e in m)           # Proporción de programas en la muestra
    return round(1 - sum(abs(cp[p]/n - cm.get(p, 0)/k) for p in cp), 4)  # 1 menos la divergencia total


st, _ = buscar(C1, lim=None, v=False)               # Busca todas las soluciones válidas de C1
sc    = [balance(m) for m in st]                    # Calcula el score de balance para cada solución
best  = st[sc.index(max(sc))]                       # Selecciona la muestra con el mayor score

print(f"Muestras válidas encontradas : {len(st)}")
print(f"Score máximo de balance      : {max(sc)}")
print(f"Score promedio               : {round(sum(sc) / len(sc), 4)}")
print(f"Score mínimo                 : {min(sc)}")
print()
print('Muestra más representativa del pool:')
print(pd.DataFrame(best).to_string(index=False))